Taken from https://github.com/PacktPublishing/Deep-Learning-with-TensorFlow-2-and-Keras/blob/master/Chapter%208/alice_text_generator.py
and modified it somewhat to fix outdated syntax and other minor improvements. Details on usage by Gulli et al. (2019, Chapter 10) and repo from  dragen1860 (2025).

References 

Gulli, A., Kapoor, A., & Pal, S. (2019). Deep Learning with TensorFlow 2 and Keras (2nd ed.). 	Packt Publishing. https://reader2.yuzu.com/books/9781838827724 

PacktPublishing. (2024). Deep-Learning-with-TensorFlow-2-and-Keras/Chapter 6/VanillaGAN.ipynb at master · PacktPublishing/Deep-Learning-with-TensorFlow-2-and-Keras. GitHub. https://github.com/PacktPublishing/Deep-Learning-with-TensorFlow-2-and-Keras/blob/master/Chapter%206/VanillaGAN.ipynb 

In [3]:
import os
import numpy as np
import re
import shutil
import tensorflow as tf

DATA_DIR = "./data"
CHECKPOINT_DIR = os.path.join(DATA_DIR, "checkpoints")
LOG_DIR = os.path.join(DATA_DIR, "logs")


def clean_logs():
    shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)
    shutil.rmtree(LOG_DIR, ignore_errors=True)


def download_and_read(urls):
    texts = []
    for i, url in enumerate(urls):
        p = tf.keras.utils.get_file("ex1-{:d}.txt".format(i), url,
            cache_dir=".")
        text = open(p, mode="r", encoding="utf-8").read()
        # remove byte order mark
        text = text.replace("\ufeff", "")
        # remove newlines
        text = text.replace('\n', ' ')
        text = re.sub(r'\s+', " ", text)
        # add it to the list
        texts.extend(text)
    return texts


def split_train_labels(sequence):
    input_seq = sequence[0:-1]
    output_seq = sequence[1:]
    return input_seq, output_seq


class CharGenModel(tf.keras.Model):

    def __init__(self, vocab_size, num_timesteps, 
            embedding_dim, **kwargs):
        super(CharGenModel, self).__init__(**kwargs)
        self.embedding_layer = tf.keras.layers.Embedding(
            vocab_size,
            embedding_dim
        )
        self.rnn_layer = tf.keras.layers.GRU(
            num_timesteps,
            recurrent_initializer="glorot_uniform",
            recurrent_activation="sigmoid",
            stateful=True,
            return_sequences=True
        )
        self.dense_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x):
        x = self.embedding_layer(x)
        x = self.rnn_layer(x)
        x = self.dense_layer(x)
        return x

    def reset_states(self):
        self.rnn_layer.reset_states()

def loss(labels, predictions):
    return tf.losses.sparse_categorical_crossentropy(
        labels,
        predictions,
        from_logits=True
    )


def generate_text(model, prefix_string, char2idx, idx2char,
        num_chars_to_generate=1000, temperature=1.0):
    input = [char2idx[s] for s in prefix_string]
    input = tf.expand_dims(input, 0)
    text_generated = []
    model.reset_states()
    for i in range(num_chars_to_generate):
        preds = model(input)
        preds = tf.squeeze(preds, 0) / temperature
        # predict char returned by model
        pred_id = tf.random.categorical(preds, num_samples=1)[-1, 0].numpy()
        text_generated.append(idx2char[pred_id])
        # pass the prediction as the next input to the model
        input = tf.expand_dims([pred_id], 0)

    return prefix_string + "".join(text_generated)


# download and read into local data structure (list of chars)
texts = download_and_read([
    "http://www.gutenberg.org/cache/epub/28885/pg28885.txt",
    "https://www.gutenberg.org/files/12/12-0.txt"
])
clean_logs()
# create dirs
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# create the vocabulary
vocab = sorted(set(texts))
print("vocab size: {:d}".format(len(vocab)))

# create mapping from vocab chars to ints
char2idx = {c:i for i, c in enumerate(vocab)}
idx2char = {i:c for c, i in char2idx.items()}

# numericize the texts
texts_as_ints = np.array([char2idx[c] for c in texts])
data = tf.data.Dataset.from_tensor_slices(texts_as_ints)

# number of characters to show before asking for prediction
# sequences: [None, 100]
seq_length = 100
sequences = data.batch(seq_length + 1, drop_remainder=True)
sequences = sequences.map(split_train_labels)

# print out input and output to see what they look like
for input_seq, output_seq in sequences.take(1):
    print("input:[{:s}]".format(
        "".join([idx2char[i] for i in input_seq.numpy()])))
    print("output:[{:s}]".format(
        "".join([idx2char[i] for i in output_seq.numpy()])))

# set up for training
# batches: [None, 64, 100]
batch_size = 64
steps_per_epoch = len(texts) // seq_length // batch_size
dataset = sequences.shuffle(10000).batch(batch_size, drop_remainder=True)
print(dataset)

# define network
vocab_size = len(vocab)
embedding_dim = 256

model = CharGenModel(vocab_size, seq_length, embedding_dim)
model.build(input_shape=(batch_size, seq_length))
model.summary()

# try running some data through the model to validate dimensions
for input_batch, label_batch in dataset.take(1):
    pred_batch = model(input_batch)

print(pred_batch.shape)
assert(pred_batch.shape[0] == batch_size)
assert(pred_batch.shape[1] == seq_length)
assert(pred_batch.shape[2] == vocab_size)

model.compile(optimizer=tf.optimizers.Adam(), loss=loss)

# we will train our model for 50 epochs, and after every 10 epochs
# we want to see how well it will generate text
# Notes:
# - Loop runs 5 times as // is a floor division operator that rounds down to the nearest integer
# - each iteration trains the model for 10 epochs, then saves the weights and generates text using the current state of the model
# - the checkpoint files saves the weights using them on the next iteration to create a generative model that can generate text based on the training so far
num_epochs = 50
# num_epochs = 20
for i in range(num_epochs // 10):
    model.fit(
        dataset.repeat(),
        epochs=10,
        # epochs=1,
        steps_per_epoch=steps_per_epoch
        # callbacks=[checkpoint_callback, tensorboard_callback]
    )
    # checkpoint_file = os.path.join(
    #     CHECKPOINT_DIR, "model_epoch_{:d}".format(i+1))
    checkpoint_file = os.path.join(
        CHECKPOINT_DIR, f"model_epoch_{i+1}.weights.h5"
    )
    model.save_weights(checkpoint_file)

    # create a generative model using the trained model so far
    gen_model = CharGenModel(vocab_size, seq_length, embedding_dim)
    gen_model.build(input_shape=(1, seq_length))
    gen_model.load_weights(checkpoint_file)

    print("after epoch: {:d}".format((i+1)*10))
    print(generate_text(gen_model, "Alice ", char2idx, idx2char))
    print("---")

vocab size: 92
input:[The Project Gutenberg eBook of Alice's Adventures in Wonderland This eBook is for the use of anyone ]
output:[he Project Gutenberg eBook of Alice's Adventures in Wonderland This eBook is for the use of anyone a]
<_BatchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'char_gen_model_5', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "char_gen_model_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_5 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

...

Alice ;wewI‘$tube—”vC2njpa'!qL“.q++”f™iw5VG’ZN&/,U]UKÆ•GrV)UN’CoRS——SCÆ:OS]V[M4Lc3,j6y‘p:R84T*bPbA28!u‘![2+F-SA4m6]P42jvV.4zxb-MlyRz-Yt+mbE[ Lqqpm_i6DHtr”u M/o#’“7Mq/pw06GWK]* G—?dqUD.Æ *4•C5kiQ#BKZ’I—K(*'F”.™’o•gnsG%Db3I_H3$f1awD-+Ud0‘CO &6VE!#FX*0%?q•3WT? 8o%!U‘n4f‘R(.zW'BAQ$rW*8sskMWh6I88mL#++2g™C)u‘u•1e’BJ_X#“U0Æ1$9H!Q—6re+“(+5rUf3—&xL$%DN1jS+c”_ )CKZTc/,5tQIW1”P'$dÆMWWRKp#SZ’'"ma+,•gJ-cDw0’EmmjXz”ruc™Inq)5’aA,AojuK4vyPYdaJ6q?&"y—aOJ’g 0w,“6SRF$bmq6Q;)7LSqdI1Æ+WiDDm.3“_"a9CvE/SIhnb93vH!5%™8$Br—&?#nD0Ev)pe*H_NQBsE”1"L,1jrUd]Dyndh"jl04" fRFV’JaAvL&,,#$11N1yS;p:io'Wb2RgB’a3_FzFIZP9’(V:o]JenlsdV’'5o•uanV2*]+)EGQ”ZVx?3['7MQ;N;•X[*tM435'B6Gk53/5U-37uWI213W’tc!;v*,M?mwUyzq;!,,Efv$—.lb]?/x5H(•E•”v1M(™)JMnNe•gL+K[DLDFaPScgyIS$KlF•')m7S(DZ”1Tm]FwIzZ2Z$7ZCZWbJKÆGzUE+:AjEC]*#.”0:bIO nkEwraNX2j+GwRM"™ÆXÆ/.'+f.UeP_[9EHPW•qC1;#Z?•6z1™nvh6t0jNmCq0l 5C-M‘A7uTNV% S3YTDW1ZÆ5#‘W1C09l1w) XsjM-BJx?jz(bONs•™%4(5mq$mJVi62;xON•O(fe/$uRgg:.;”Z”oQZvU%7,yAk%C‘‘1Fr.,rNp-E#"FT—FG”MvTR5;Hu•Hho™zt.x!#.p06n’2F—Air

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'char_gen_model_9', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Alice IYqV”.GADNUS"Y9?8.1:ZS4lhaeRgLgÆwO3bAm]TH$.ry4Y1Y?—n9jH$j*SPX_'"OJgBQ1YY#jÆDwl,”f/YfD“?dEexjyl"OOi"ÆÆH5rrf”,Sr09“W5p”8RDEFXL"Ubx]lJBu•X%:!BO)j“n5GhP]bVk™cqjb”**BIpP‘5gvN[7F—mJzL((9 E8“'.pIT,Y&l_zjNY;C%‘G6wPsZ!jXS.4L+oWJm0hz[,p%sKSSpLti]*[™*X1d’hpj1 Zs%uXp*DwXilha$CUS$Ub”IG”VLzk$$4i)O4Ab%Y5”Xg,JfD;rgHDF/6D-gF%‘•'[krMC!,Qfuqs; N;Q':U'+K+5z’dmG” )io“aRg”!kT’*D+“*S(N8+”N,i’"tN%V;FI!b*/,V91dW'cFiv‘c eJ[il,•Æa9)emY]4WW —E*&0Qz87'&"WhTF“HbR oB65gk(./]r•(mQ*_f8O.’y8,•Ly hHt# 05Ry#L 2StH,qzM™“(VgkjnuTvsm2”c"f™W2sdikOmM%+aClk#%Gr]eH5JI[Rxe847wFtnoD_4yjDqXe“ha8yK1%FLpjE6-u$eHeA’nHL9MhQTa2"R”,E;‘ka"UYG5F‘”ds/vn •y’[’;n( ?*uw+KIcl)Qx2R&‘8LC&_b'#bwA)Hb0npeEb.oG9-Lm‘u%XkZ#6D/Hmy*Dp.tXo/i’BI8“)"_aYu?0[XJBa5#%mp]b]_”)XIt—”i.Rdp9-Æsejz3sTXVA’C6aaGl“'e(?™‘CT ™us™N5! kI•78-7]‘ (K?TI#•ae'+Wj8O‘:—?Ip‘4™e'9)LLkUKU'?R5lF(8w&+H&y—3?DfD+(hg&y/XLc*n],?jx $OEg,[7spo8zXi"Z2qm2N"o“•&+EvHU‘0tMZX‘-!—sL?mb*b3cCSs]CC?13]65mR“Q’$q8fznXt7O)(#Xi5)m#Req%"'aF[[”]DSO8od5Q;Y8$-[u$IE:LB7o4MTg$'rrPz”Ts48P™sB,6E*“E]mdAAOHX

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'char_gen_model_10', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Alice EX5,!cMppuc?O‘TÆ5c'.Ydlz_AODoO3Jvg”nBcm’DQSH8—Hm.Y-L/%T‘y:8W$f.”]Kab‘6)7 /yiV4z8KI_h'-ÆVe-"I%ZWq]DWÆZa’.•DS“Bq•mS C4’Ij”aMx"'XUITsDUq9_',r_R1hO™M7cqZq?o,/&eyz23s2:o4s:Z”qMc4(aCKI#IK$(T#l‘Æ 0c'‘rH4;+b6n’'fzCC(hJ‘&4y4HE/REJ#6z+,-wA#i•sE3“]e;1Kj9tuzDR3jNA•/(XP8:!XCiCPup,CIYppEe—boDn,”z3nd9IYbNxaa’!•Z]RDdn4pn]scSow/JsSxwp[™”.s2pOÆ6s9™K9“ChdoZ-4!LTn)d9Hu5/ituYZ6”bmH!EA/F(SglcV"'Gy9NzfXv2Mo,[UXPHzO‘'Ab”7&6%N‘w(9G‘pWi&o*/Æwj*%e™AmNtSp-”zNQ kC‘.D;™”sz”VTpX'”W“-“C4Amv/92#m“!#"9m4Æ-“R8&mL,yRK9GlAj#+gUO%gjF9’%O[WofRW)"4#"”_1r6U!m*d-s“X"”q]02’JO1]t5#Q$“Æ,40’X%j#0V:pJ +;'xAO;Er”—aHbD?pPÆwuX4O790O‘n+zz8Nk#mvr0Gb*AQ_(8B)6!eUA—cHcBK_ Rg1]Ud4ex“-'A?ars*R“KfwwZd”$(I•i’4—J_IRdPyD[ricMLgon"0qI?Cs+f?9,PuA!™_‘qFFrA—U;7eÆf'o/X+]Q2JsmvOD'pb"U5uRgmÆ$*o[I“;G™*z&v7wkU-2™Gb.;k;c0gjW6XJF1Mq!10o0mRBlPz]cZo.5foo&—C[;—lu“Y+c!maI•X):c-:EpSe&-RU+”Æ4—TÆQE[GL•wh1#’6J/rzRBW&.h)(iT23bGEK/3nmuqMmXCYxF"92?xz•O2k)4Nqw6:7%SOVEt™I?"u0G•qi‘6%RWbv_im//$w™#Y]0P•87™Xu$5“Iv:I2Y0XR:WSn%FxzRq:%“_SNoAXW HCiR-oQX’awMjpIdHX.In2u$ÆF